## The Elbow method

In [7]:
#imports
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import psycopg2
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv
load_dotenv('/home/luifer/Numbersdontlie/Data-Science-Piscine/day2/ex00/.env', override=True)
import os

In [4]:
user = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
dbname = os.getenv('POSTGRES_DB')

uri = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"

In [5]:
# generate the query to get # of customers monthly
query = """
SELECT *
FROM customers
WHERE event_type = 'purchase'
ORDER BY event_time ASC;
"""

In [8]:
# generate the dataframe
with psycopg2.connect(
    user=user,
    password=password,
    host=host,
    port=port,
    dbname=dbname,
) as conn:
    df_purchase = pl.read_database(
        query,
        conn,
        infer_schema_length=10_000,
        schema_overrides={
            "product_id": pl.Int64,
            "user_id": pl.Int64,
            "price": pl.Float64,
        },
    )

In [9]:
# feature preprocessing
# 1. Feature Preprocessing (using customer_df from previous steps)
# Log transformation handles skewed distribution (common in RFM data)
features = df_purchase.select(
    [
        pl.col("frequency").log1p(),
        pl.col("monetary_value").log1p(),
    ]
).to_numpy()

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

ColumnNotFoundError: unable to find column "frequency"; valid columns: ["event_time", "event_type", "product_id", "price", "user_id", "user_session", "category_id", "category_code", "brand"]